# Baseline: VGG16-BN

## 1. Imports

In [ ]:
import os
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

import torchvision
from torchvision import transforms
from torchvision.transforms import functional as TF

from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

try:
    import yaml
except Exception:
    yaml = None



## 2. Shared Utilities (Unified Interface)

In [ ]:
def set_seed(seed: int):
    # Fix randomness for reproducibility.
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


class ResizeLongestSidePadSquare:
    # Resize by the longest side and pad to a centered square.
    def __init__(self, size: int, fill: int = 0):
        self.size = int(size)
        self.fill = int(fill)

    def __call__(self, img: Image.Image):
        w, h = img.size
        if w <= 0 or h <= 0:
            raise ValueError('invalid image size')

        scale = self.size / max(w, h)
        new_w = max(1, int(round(w * scale)))
        new_h = max(1, int(round(h * scale)))
        img = TF.resize(img, [new_h, new_w], interpolation=transforms.InterpolationMode.BILINEAR)

        pad_w = self.size - new_w
        pad_h = self.size - new_h
        left = pad_w // 2
        top = pad_h // 2
        right = pad_w - left
        bottom = pad_h - top
        return TF.pad(img, [left, top, right, bottom], fill=self.fill)


class ManifestImageDataset(Dataset):
    # Read samples only from split_all.csv instead of scanning directories directly.
    def __init__(self, dataset_root: Path, df: pd.DataFrame, class_to_idx: dict, transform=None):
        self.dataset_root = Path(dataset_root)
        self.transform = transform
        self.class_to_idx = dict(class_to_idx)
        self.samples = []

        for _, r in df.iterrows():
            rel_path = str(r['path'])
            cls = str(r['class'])
            if cls not in self.class_to_idx:
                continue
            abs_path = self.dataset_root / rel_path
            if abs_path.exists():
                self.samples.append((abs_path, self.class_to_idx[cls]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        p, y = self.samples[idx]
        img = Image.open(p).convert('RGB')
        if self.transform is not None:
            img = self.transform(img)
        return img, y


def evaluate_classifier(model, loader, device):
    # Unified evaluation interface.
    model.eval()
    ys, ps = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device, non_blocking=True)
            logits = model(x)
            pred = logits.argmax(1).cpu().numpy()
            ys.append(y.numpy())
            ps.append(pred)

    y_true = np.concatenate(ys) if ys else np.array([])
    y_pred = np.concatenate(ps) if ps else np.array([])
    if len(y_true) == 0:
        return 0.0, 0.0, y_true, y_pred

    acc = float(accuracy_score(y_true, y_pred))
    macro_f1 = float(f1_score(y_true, y_pred, average='macro'))
    return acc, macro_f1, y_true, y_pred


def save_outputs(out_dir: Path, config: dict, history: list, metrics: dict, y_true, y_pred, classes):
    # Unified output format.
    out_dir.mkdir(parents=True, exist_ok=True)
    (out_dir / 'config.json').write_text(json.dumps(config, indent=2, ensure_ascii=False), encoding='utf-8')
    (out_dir / 'train_history.json').write_text(json.dumps(history, indent=2, ensure_ascii=False), encoding='utf-8')
    (out_dir / 'metrics.json').write_text(json.dumps(metrics, indent=2, ensure_ascii=False), encoding='utf-8')
    (out_dir / 'test_report.json').write_text(json.dumps({'test': metrics['test']}, indent=2, ensure_ascii=False), encoding='utf-8')

    if len(y_true) > 0:
        cm = confusion_matrix(y_true, y_pred, labels=list(range(len(classes))))
        pd.DataFrame(cm, index=classes, columns=classes).to_csv(out_dir / 'confusion_matrix.csv', encoding='utf-8')



## 3. Model Builder

In [ ]:
MODEL_KEY = 'vgg16_bn'
def build_model(num_classes: int, use_pretrained: bool = True):
    weights = torchvision.models.VGG16_BN_Weights.IMAGENET1K_V1 if use_pretrained else None
    m = torchvision.models.vgg16_bn(weights=weights)
    in_features = m.classifier[-1].in_features
    m.classifier[-1] = nn.Linear(in_features, num_classes)
    return m



## 4. Configuration and Paths (from `data_prep`)

In [ ]:
# Unified path configuration
PROJECT_ROOT = Path.cwd().resolve().parent.parent
DATASET_ROOT = PROJECT_ROOT / 'dataset'
SPLIT_DIR = PROJECT_ROOT / 'outputs' / 'data_prep'
SPLIT_ALL_PATH = SPLIT_DIR / 'split_all.csv'
PREPROCESS_YAML = SPLIT_DIR / 'preprocess_config.yaml'
PREPROCESS_JSON = SPLIT_DIR / 'preprocess_config.json'

BASE_OUT = PROJECT_ROOT / 'outputs' / 'baseline'
OUT_DIR = BASE_OUT / MODEL_KEY

# Unified training hyperparameters
SEED = 42
BATCH_SIZE = 128
EPOCHS = 20
LR = 1e-3
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 4
USE_PRETRAINED = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
set_seed(SEED)

if not SPLIT_ALL_PATH.exists():
    raise FileNotFoundError(f'split file not found: {SPLIT_ALL_PATH}')

# Load the preprocessing protocol from `data_prep`
if PREPROCESS_YAML.exists() and yaml is not None:
    preprocess_cfg = yaml.safe_load(PREPROCESS_YAML.read_text(encoding='utf-8'))
elif PREPROCESS_JSON.exists():
    preprocess_cfg = json.loads(PREPROCESS_JSON.read_text(encoding='utf-8'))
else:
    preprocess_cfg = {
        'img_size': 224,
        'pad_fill': 0,
        'normalize': {'mean': [0.485, 0.456, 0.406], 'std': [0.229, 0.224, 0.225]},
        'train_augment': {
            'random_horizontal_flip': 0.5,
            'color_jitter': {'brightness': 0.2, 'contrast': 0.2, 'saturation': 0.2, 'hue': 0.02}
        }
    }

IMG_SIZE = int(preprocess_cfg.get('img_size', 224))
PAD_FILL = int(preprocess_cfg.get('pad_fill', 0))
NORM_MEAN = tuple(preprocess_cfg.get('normalize', {}).get('mean', [0.485, 0.456, 0.406]))
NORM_STD = tuple(preprocess_cfg.get('normalize', {}).get('std', [0.229, 0.224, 0.225]))
AUG = preprocess_cfg.get('train_augment', {})
FLIP_P = float(AUG.get('random_horizontal_flip', 0.5))
CJ = AUG.get('color_jitter', {'brightness': 0.2, 'contrast': 0.2, 'saturation': 0.2, 'hue': 0.02})

print('[INFO] DEVICE:', DEVICE)
print('[INFO] SPLIT_ALL_PATH:', SPLIT_ALL_PATH)
print('[INFO] OUT_DIR:', OUT_DIR)



## 5. Training and Unified Outputs

In [ ]:
# Build train/val/test only from split_all.csv
split_df = pd.read_csv(SPLIT_ALL_PATH)
required_cols = {'path', 'split', 'class'}
if not required_cols.issubset(set(split_df.columns)):
    raise ValueError(f'split_all.csv missing fields: {required_cols}')

classes = sorted(split_df['class'].dropna().unique().tolist())
class_to_idx = {c: i for i, c in enumerate(classes)}

train_df = split_df[split_df['split'] == 'train'].copy().reset_index(drop=True)
val_df = split_df[split_df['split'] == 'val'].copy().reset_index(drop=True)
test_df = split_df[split_df['split'] == 'test'].copy().reset_index(drop=True)

# Unified transforms (parameters come from `data_prep`)
base_resize_pad = ResizeLongestSidePadSquare(IMG_SIZE, fill=PAD_FILL)
train_tfms = transforms.Compose([
    base_resize_pad,
    transforms.RandomHorizontalFlip(p=FLIP_P),
    transforms.ColorJitter(
        brightness=float(CJ.get('brightness', 0.2)),
        contrast=float(CJ.get('contrast', 0.2)),
        saturation=float(CJ.get('saturation', 0.2)),
        hue=float(CJ.get('hue', 0.02)),
    ),
    transforms.ToTensor(),
    transforms.Normalize(NORM_MEAN, NORM_STD),
])

eval_tfms = transforms.Compose([
    base_resize_pad,
    transforms.ToTensor(),
    transforms.Normalize(NORM_MEAN, NORM_STD),
])

train_ds = ManifestImageDataset(DATASET_ROOT, train_df, class_to_idx, transform=train_tfms)
val_ds = ManifestImageDataset(DATASET_ROOT, val_df, class_to_idx, transform=eval_tfms)
test_ds = ManifestImageDataset(DATASET_ROOT, test_df, class_to_idx, transform=eval_tfms)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print('[INFO] classes:', classes)
print('[INFO] train/val/test:', len(train_ds), len(val_ds), len(test_ds))

model = build_model(num_classes=len(classes), use_pretrained=USE_PRETRAINED).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

history = []
best_f1 = -1.0
best_state = None

for ep in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    total_n = 0

    for x, y in train_loader:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.size(0)
        total_n += x.size(0)

    scheduler.step()
    train_loss = total_loss / max(1, total_n)
    val_acc, val_f1, _, _ = evaluate_classifier(model, val_loader, DEVICE)

    history.append({
        'epoch': ep,
        'train_loss': float(train_loss),
        'val_acc': float(val_acc),
        'val_macro_f1': float(val_f1),
        'lr': float(optimizer.param_groups[0]['lr'])
    })

    print(f"[Epoch {ep:02d}/{EPOCHS}] train_loss={train_loss:.4f} val_acc={val_acc:.4f} val_macro_f1={val_f1:.4f}")

    if val_f1 > best_f1:
        best_f1 = val_f1
        best_state = {k: v.cpu() for k, v in model.state_dict().items()}

if best_state is not None:
    model.load_state_dict(best_state)

OUT_DIR.mkdir(parents=True, exist_ok=True)
torch.save({'model_key': MODEL_KEY, 'best_val_macro_f1': float(best_f1), 'model_state_dict': model.state_dict()}, OUT_DIR / 'best.pt')

test_acc, test_f1, y_true, y_pred = evaluate_classifier(model, test_loader, DEVICE)

config = {
    'model_key': MODEL_KEY,
    'seed': SEED,
    'batch_size': BATCH_SIZE,
    'epochs': EPOCHS,
    'lr': LR,
    'weight_decay': WEIGHT_DECAY,
    'num_workers': NUM_WORKERS,
    'use_pretrained': USE_PRETRAINED,
    'img_size': IMG_SIZE,
    'pad_fill': PAD_FILL,
    'normalize_mean': list(NORM_MEAN),
    'normalize_std': list(NORM_STD),
    'flip_p': FLIP_P,
    'color_jitter': {
        'brightness': float(CJ.get('brightness', 0.2)),
        'contrast': float(CJ.get('contrast', 0.2)),
        'saturation': float(CJ.get('saturation', 0.2)),
        'hue': float(CJ.get('hue', 0.02)),
    }
}

metrics = {
    'best_val_macro_f1': float(best_f1),
    'test': {'acc': float(test_acc), 'macro_f1': float(test_f1)}
}

save_outputs(OUT_DIR, config, history, metrics, y_true, y_pred, classes)
print('[DONE] saved to', OUT_DIR)

